In [ ]:
import langchain
from langchain_community.document_loaders import TextLoader,PyPDFLoader,WebBaseLoader

In [ ]:
text_loader = TextLoader("files/test.txt")
text_docs = text_loader.load()
text_docs

In [ ]:
pdf_loader = PyPDFLoader("files/gigr.pdf")
pdf_docs = pdf_loader.load()
pdf_docs

In [ ]:
import bs4
web_loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence",
                           bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("mw-page-title-main","vector-body-before-content","mw-body-content"))))

page_title = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence",
                           bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("mw-page-title-main")))).load()[0].page_content
content = web_loader.load()
content


In [ ]:
with open(f"files/artificial_intelligence.txt","w",encoding='utf-8') as f:
    f.write(content[0].page_content)

### text splitter

In [ ]:
from langchain_text_splitters import (RecursiveCharacterTextSplitter,
                                      CharacterTextSplitter,
                                      HTMLHeaderTextSplitter,
                                      RecursiveJsonSplitter,)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_pdf_docs = text_splitter.split_documents(pdf_docs)
final_text_docs = text_splitter.split_documents(text_docs)
final_wiki_docs = text_splitter.split_documents(content)
final_wiki_docs

In [ ]:
# character splitter
character_splitter = CharacterTextSplitter(separator='\n',)
character_splitter.split_documents(text_docs)

### html splitter

In [ ]:
split_list = [
    ("h1","header 1"),
    ("h2","header 2"),
    ("h3","header 3"),
    ("h4","header 4"),
]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=split_list)
html_splitter.split_text_from_url("https://docs.langchain.com/oss/python/langchain/overview")

In [ ]:
# Json Splitter
json_splitter = RecursiveJsonSplitter(max_chunk_size=1000,)

### json splitter

In [ ]:
json_url = "https://api.smith.langchain.com/openapi.json"
import requests
import json
response = requests.get(json_url).json()

with open("files/openapi.json","w",encoding='utf-8') as f:
    json.dump(response,f,indent=4)

In [ ]:
json_splitter.split_json(response)

## embedding and storing vector

In [ ]:
from langchain_community.vectorstores import Chroma,FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os

os.environ['HF_TOKEN'] = os.getenv("HUGGING_FACE_TOKEN")

In [ ]:
hugging_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",)


# db = FAISS.from_documents(documents=final_pdf_docs, embedding=hugging_embeddings)
db = Chroma(collection_name="text_docs_collection_2",
            persist_directory="chromadb_text_docs",)

In [ ]:
db.add_documents(documents=final_pdf_docs, embedding=hugging_embeddings)
# print(db._collection.count())
db.add_documents(documents=final_text_docs, embedding=hugging_embeddings)
# print(db._collection.count())
db.add_documents(documents=final_wiki_docs, embedding=hugging_embeddings)


In [ ]:
test_str = "what is use of artificial intelligence"
docs = db.similarity_search(test_str)
for i in range(len(docs)):
    print(f"[chuck no {i}] -- \n {docs[i].page_content}",end='\n')

In [ ]:
reviter = db.as_retriever()
reviter.invoke(test_str)

In [ ]:
hugging_embeddings.embed_query("iam")

### retiver for vector database

In [ ]:
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

str_output_parser = StrOutputParser()
os.environ["GOOGLE_API_KEY"] = ""
os.environ["GROQ_API_KEY"] = ""


In [ ]:
# gemini = GoogleGenerativeAI(
#     model="gemini-3-flash-preview",
#     temperature=1.0,  # Gemini 3.0+ defaults to 1.0
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )

llm = ChatGroq(model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [ ]:
response = llm.invoke("i am stranger and response only in three words")

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You are content writer for my portfolio website. i give you raw content in my template, change them according to the AI and ML student or fresher's portfolio website"),
    ("user","{input}")
])
prompt

In [ ]:
content_prompt = ChatPromptTemplate.from_template(
    """
        You are content writer for my portfolio website. i give you raw content in my template,
          change them according to the AI and ML student or fresher's portfolio website with given raw content context and also my index.html file content as reference.

          <raw content context>
          {context}
          </raw content context>

          <index.html content>
            {index_content}
            </index.html content>
    """
)

In [ ]:
chain = content_prompt|llm|str_output_parser
chain

In [ ]:
from langchain_community.document_loaders import BSHTMLLoader,TextLoader

web_loader = BSHTMLLoader('files/index.html')
text_loader = TextLoader("files/index.html")
text_loader.load()


In [ ]:
with open("files/index_raw.txt","w+",encoding='utf-8') as f:
    f.write(web_loader.load()[0].page_content)

In [ ]:
content_reponse = chain.invoke({"context":content[0].page_content,"index_content":web_loader.load()[0].page_content})

In [ ]:
with open("files/index_response.txt","w+",encoding='utf-8') as f:
    f.write(content_reponse)